# Phase 5 fix — Step 2: Detect-then-crop (TEST ONLY, no retraining)

The disease model looks at background, not the leaf, on cluttered PlantDoc images (72% acc). Two training fixes failed (background randomization, LP-FT). This notebook tests the literature's strongest lever — **crop the leaf, then classify** — using the **existing OLD model** (no retraining).

We measure 3 ways on the PlantDoc test set:

| Method | What it is |
|---|---|
| **No crop** | today's model on the full image (~72%) |
| **YOLO auto-crop** | a pretrained leaf detector finds the leaf, we crop + classify → the deployable fix |
| **GT-box crop** | crop by PlantDoc's human-drawn boxes → the ceiling cropping can reach (proves the idea) |

Decision: if **YOLO-crop** recovers accuracy → add YOLO crop to the pipeline (no retraining). If only **GT-crop** helps → retrain on cropped images. If neither → cropping isn't the fix.

Reuses `src/disease/detect_crop.py` (VOC parse + crop + class match; unit-tested) and the existing `DiseaseInferenceEngine`.

In [ ]:
# Cell 2 — clone repo + the PlantDoc detection (box) repo + deps + HF login + GPU.
import os, shutil, subprocess, sys

REPO_PATH = "/content/iks-rag-thesis"
REPO_URL = "https://github.com/ankit8453/iks-rag-thesis.git"
DET_PATH = "/content/PlantDoc-Object-Detection-Dataset"
DET_URL = "https://github.com/pratikkayal/PlantDoc-Object-Detection-Dataset.git"

os.chdir("/content")
shutil.rmtree(REPO_PATH, ignore_errors=True)
env = os.environ.copy(); env["GIT_LFS_SKIP_SMUDGE"] = "1"
r = subprocess.run(["git", "clone", REPO_URL, REPO_PATH], env=env, capture_output=True, text=True)
if r.returncode != 0:
    print(r.stdout); print(r.stderr); raise RuntimeError("clone iks failed")

# PlantDoc detection repo = the human-drawn boxes (the 'answer key').
if not os.path.isdir(DET_PATH):
    r = subprocess.run(["git", "clone", "--depth", "1", DET_URL, DET_PATH], env=env, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout); print(r.stderr); raise RuntimeError("clone detection repo failed")

os.chdir(REPO_PATH); sys.path.insert(0, REPO_PATH)
print("repos ready")

DEPS = [
    "timm>=1.0", "datasets>=2.20", "huggingface_hub>=0.24",
    "grad-cam>=1.5", "ultralytics>=8.0", "pydantic>=2.7",
    "opencv-python-headless", "matplotlib>=3.7",
]
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *DEPS], capture_output=True, text=True)
if r.returncode != 0:
    print("\n".join(r.stdout.splitlines()[-30:])); print("\n".join(r.stderr.splitlines()[-30:]))
    raise RuntimeError("pip failed")
print("deps installed")

from huggingface_hub import HfApi, login
login(); print("HF user:", HfApi().whoami().get("name"))
import torch
assert torch.cuda.is_available(), "Switch to T4 GPU."
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Cell 3 — locate the detection TEST folder + load the OLD classifier + class map.
import glob, json, os
from src.disease.infer import DiseaseInferenceEngine
from src.disease.detect_crop import (
    parse_voc_xml, crop_to_box, build_normalized_class_map, match_class_to_index,
)

# Find the folder holding the test .xml files (repo layout: TEST/ with .jpg + .xml).
cands = ["/content/PlantDoc-Object-Detection-Dataset/TEST",
         "/content/PlantDoc-Object-Detection-Dataset/test"]
DET_TEST = next((c for c in cands if os.path.isdir(c)), None)
if DET_TEST is None:  # fall back: search for any dir with xml files
    for root, _dirs, files in os.walk("/content/PlantDoc-Object-Detection-Dataset"):
        if any(f.endswith('.xml') for f in files) and ('test' in root.lower()):
            DET_TEST = root; break
assert DET_TEST, "Could not find the detection TEST folder"
xmls = sorted(glob.glob(os.path.join(DET_TEST, '*.xml')))
print(f"detection TEST dir: {DET_TEST}  ({len(xmls)} xml files)")

eng = DiseaseInferenceEngine(model_source="ankit-iiitdmj/iks-disease-plantdoc", device="cuda")
with open("data/splits/plantdoc/class_map.json") as f:
    class_map = json.load(f)
norm_map = build_normalized_class_map(class_map)
print(f"classifier: {eng.num_classes} classes")

In [ ]:
# Cell 4 — PATH A: crop by GROUND-TRUTH boxes (the ceiling / proof).
# For every human-drawn box: crop -> classify -> compare to the box's class.
from PIL import Image

def img_for_xml(xml_path):
    base = os.path.splitext(xml_path)[0]
    for ext in (".jpg", ".jpeg", ".png", ".JPG", ".PNG"):
        if os.path.exists(base + ext):
            return base + ext
    return None

gt_correct = gt_total = unmatched = no_img = 0
gt_samples = []  # (crop, pred_name, gt_name) for a few previews
for xml in xmls:
    ipath = img_for_xml(xml)
    if ipath is None:
        no_img += 1; continue
    try:
        pil = Image.open(ipath).convert("RGB")
    except Exception:
        no_img += 1; continue
    for b in parse_voc_xml(xml):
        gt_idx = match_class_to_index(b.name, norm_map)
        if gt_idx is None:
            unmatched += 1; continue
        crop = crop_to_box(pil, b, pad_frac=0.10)
        pred = eng.predict(crop).prediction
        ok = int(pred.class_index) == gt_idx
        gt_correct += int(ok); gt_total += 1
        if len(gt_samples) < 6:
            gt_samples.append((crop, pred.class_name, b.name, ok))

gt_acc = gt_correct / max(1, gt_total)
print(f"GT-box crop accuracy: {gt_acc:.1%}  ({gt_correct}/{gt_total})")
print(f"  (unmatched class names: {unmatched}, missing images: {no_img})")

In [ ]:
# Cell 5 — PATH B: pretrained YOLO leaf detector -> crop -> classify (the deployable fix).
#          PATH C: no crop (full image) baseline, on the SAME images for a fair compare.
from huggingface_hub import hf_hub_download, list_repo_files
from ultralytics import YOLO
from collections import Counter
from src.disease.detect_crop import normalize_class_name

# Load the pretrained YOLOv8 leaf detector (no training).
YOLO_REPO = "foduucom/plant-leaf-detection-and-classification"
wfiles = [f for f in list_repo_files(YOLO_REPO) if f.endswith(".pt")]
wname = "best.pt" if "best.pt" in wfiles else wfiles[0]
wpath = hf_hub_download(YOLO_REPO, wname)
yolo = YOLO(wpath)
print("YOLO loaded:", wname)

def image_gt_index(xml):
    """Image-level label = majority class over its boxes (PlantDoc images
    are effectively single-disease)."""
    names = [b.name for b in parse_voc_xml(xml)]
    idxs = [match_class_to_index(n, norm_map) for n in names]
    idxs = [i for i in idxs if i is not None]
    if not idxs:
        return None
    return Counter(idxs).most_common(1)[0][0]

def yolo_crop(pil):
    """Return the highest-confidence detected box crop, or the full image
    if nothing is detected."""
    res = yolo.predict(pil, verbose=False, conf=0.25)[0]
    if res.boxes is None or len(res.boxes) == 0:
        return pil, False
    confs = res.boxes.conf.tolist()
    best = max(range(len(confs)), key=lambda i: confs[i])
    x1, y1, x2, y2 = (int(v) for v in res.boxes.xyxy[best].tolist())
    if x2 <= x1 or y2 <= y1:
        return pil, False
    return pil.crop((x1, y1, x2, y2)).convert("RGB"), True

yolo_correct = raw_correct = n_img = detected = 0
yolo_samples = []
for xml in xmls:
    gt_idx = image_gt_index(xml)
    ipath = img_for_xml(xml)
    if gt_idx is None or ipath is None:
        continue
    try:
        pil = Image.open(ipath).convert("RGB")
    except Exception:
        continue
    n_img += 1
    # C: no crop
    raw_pred = eng.predict(pil).prediction
    raw_correct += int(int(raw_pred.class_index) == gt_idx)
    # B: yolo crop
    crop, found = yolo_crop(pil)
    detected += int(found)
    y_pred = eng.predict(crop).prediction
    yolo_correct += int(int(y_pred.class_index) == gt_idx)
    if len(yolo_samples) < 6:
        yolo_samples.append((pil, crop, y_pred.class_name, found))

raw_acc = raw_correct / max(1, n_img)
yolo_acc = yolo_correct / max(1, n_img)
print(f"No-crop (baseline) accuracy: {raw_acc:.1%}  ({raw_correct}/{n_img})")
print(f"YOLO-crop accuracy:          {yolo_acc:.1%}  ({yolo_correct}/{n_img})")
print(f"  (YOLO found a leaf in {detected}/{n_img} images)")

In [ ]:
# Cell 6 — RESULTS TABLE + previews + Grad-CAM (full vs GT-crop) so we SEE the attention move.
import matplotlib.pyplot as plt

print("=" * 56)
print("DETECT-THEN-CROP — PlantDoc test (OLD model, no retraining)")
print("=" * 56)
print(f"{'method':<26}{'accuracy':>10}")
print("-" * 56)
print(f"{'No crop (today)':<26}{raw_acc:>9.1%}")
print(f"{'YOLO auto-crop':<26}{yolo_acc:>9.1%}")
print(f"{'GT-box crop (ceiling)':<26}{gt_acc:>9.1%}")
print("=" * 56)
print(f"YOLO recovered {(yolo_acc-raw_acc)*100:+.1f} pp vs no-crop; "
      f"ceiling (GT) is {(gt_acc-raw_acc)*100:+.1f} pp.")

# Preview: YOLO detections (full -> crop)
if yolo_samples:
    fig, ax = plt.subplots(2, len(yolo_samples), figsize=(3*len(yolo_samples), 6))
    for i, (full, crop, name, found) in enumerate(yolo_samples):
        ax[0][i].imshow(full); ax[0][i].set_title("full", fontsize=9); ax[0][i].axis("off")
        ax[1][i].imshow(crop); ax[1][i].set_title(f"YOLO crop\n{name[:16]}" + ("" if found else " (none)"), fontsize=8); ax[1][i].axis("off")
    plt.suptitle("YOLO leaf detection: full image -> cropped leaf"); plt.tight_layout(); plt.show()

# Grad-CAM: does attention move onto the leaf after cropping?
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from src.explain.gradcam import _preprocess_for_gradcam

def cam(pil):
    t, rgb_u8, rgb_f = _preprocess_for_gradcam(pil, image_size=eng.image_size)
    mod = eng.model._module if hasattr(eng.model, "_module") else eng.model
    bb = eng.model.get_feature_extractor(); mod.eval()
    tt = t.to(next(mod.parameters()).device).requires_grad_(True)
    pred = eng.predict(pil).prediction
    g = GradCAM(model=mod, target_layers=[bb.blocks[-2]])(
        input_tensor=tt, targets=[ClassifierOutputTarget(int(pred.class_index))])[0]
    return show_cam_on_image(rgb_f, g, use_rgb=True), rgb_u8

for crop, _pname, _gname, _ok in gt_samples[:4]:
    ov, rgb = cam(crop)
    fig, ax = plt.subplots(1, 2, figsize=(8, 4))
    ax[0].imshow(rgb); ax[0].set_title("GT-cropped leaf"); ax[0].axis("off")
    ax[1].imshow(ov); ax[1].set_title("Grad-CAM on the crop"); ax[1].axis("off")
    plt.tight_layout(); plt.show()

## How to read the result

| If we see... | It means... | Next step |
|---|---|---|
| YOLO-crop ≈ GT-crop, both > no-crop | cropping is the fix AND YOLO is good enough | **Add YOLO crop to the pipeline — NO retraining.** Done. |
| GT-crop >> no-crop but YOLO-crop weak | cropping works, but the detector is the weak link | fine-tune YOLO on PlantDoc boxes, or retrain classifier on GT crops |
| GT-crop ≈ no-crop | cropping is NOT the fix | stop; the bias is elsewhere |

Report all three numbers honestly: **no-crop / YOLO-crop / GT-oracle**. The GT number is a ceiling (perfect boxes), not a deployable result.